# Week 10
### CSC101 Introduction to Programming

This week I practised using pandas to answer questions about sales and spending. I used simple examples so I could understand what each part does.

In [1]:
import pandas as pd
from pathlib import Path

# Keep the CSV files in the same folder as this notebook
sales_df = pd.read_csv("sales_data.csv")
transactions_df = pd.read_csv("transactions.csv")
print("Sales data:")
print(sales_df.head())
print("\nTransactions:")
print(transactions_df.head())

Sales data:
         Date        Product  Quantity  Unit_Price  Total_Sale Sales_Rep Region
0  2026-09-01       Notebook         8         6.5        52.0     Alice  North
1  2026-09-02           Pens        20         2.5        50.0       Bob  South
2  2026-09-03  Printer Paper        10        12.0       120.0     Alice  North
3  2026-09-04     Desk Chair         2       120.0       240.0      Cara   West
4  2026-09-05       Notebook        15         6.5        97.5       Bob  South

Transactions:
         Date   Category  Amount
0  2026-07-03  Groceries     -85
1  2026-07-07  Transport     -40
2  2026-07-12     Dining     -35
3  2026-07-22  Groceries     -75
4  2026-08-02  Groceries    -100


## Exercise 1.1 - Sales by region

**My practice AI prompt:** How do I use pandas groupby to find total sales, average sale and transaction count for each region? Please explain it simply.

I will group the rows by region and calculate the three values.

In [2]:
region_sales = sales_df.groupby("Region").agg(
    Total_Sales=("Total_Sale", "sum"),
    Average_Sale=("Total_Sale", "mean"),
    Transactions=("Total_Sale", "count")
).round(2).sort_values("Total_Sales", ascending=False)
print(region_sales)
print("\nBest region:", region_sales.index[0])

        Total_Sales  Average_Sale  Transactions
Region                                         
South         675.5        168.88             4
North         433.0         86.60             5
West          393.0        131.00             3

Best region: South


Grouping by region makes the results easier to compare. The table is enough for a quick summary, but a chart could make it easier for a manager to read.

## Exercise 1.2 - Top products

**My practice AI prompt:** Show me a simple way to find the five products with the highest revenue and units sold.

The worksheet also mentions profit margins, but we only have selling prices, not costs. So we cannot calculate actual profit margins.

In [3]:
products = sales_df.groupby("Product").agg(
    Revenue=("Total_Sale", "sum"),
    Units_Sold=("Quantity", "sum"),
    Average_Price=("Unit_Price", "mean")
).round(2)
print("Top products by revenue:")
print(products.sort_values("Revenue", ascending=False).head(5))
print("\nTop products by units:")
print(products.sort_values("Units_Sold", ascending=False).head(5))

Top products by revenue:
               Revenue  Units_Sold  Average_Price
Product                                          
Desk Chair       720.0           6          120.0
Printer Paper    384.0          32           12.0
Notebook         227.5          35            6.5
Pens             170.0          68            2.5

Top products by units:
               Revenue  Units_Sold  Average_Price
Product                                          
Pens             170.0          68            2.5
Notebook         227.5          35            6.5
Printer Paper    384.0          32           12.0
Desk Chair       720.0           6          120.0


I would check both revenue and units sold before ordering more stock. High revenue does not always mean the most units sold. I would also need stock levels and product costs before making a final decision.

## Exercise 1.3 - Sales representatives

**My practice AI prompt:** Can you show total revenue, average sale and transaction count for each sales rep using pandas?

In [4]:
rep_sales = sales_df.groupby("Sales_Rep").agg(
    Revenue=("Total_Sale", "sum"),
    Average_Sale=("Total_Sale", "mean"),
    Transactions=("Total_Sale", "count")
).round(2).sort_values("Revenue", ascending=False)
print(rep_sales)
print("\nRevenue by rep and region:")
print(sales_df.pivot_table(index="Sales_Rep", columns="Region", values="Total_Sale",
                           aggfunc="sum", fill_value=0))

           Revenue  Average_Sale  Transactions
Sales_Rep                                     
Bob          675.5        168.88             4
Alice        433.0         86.60             5
Cara         393.0        131.00             3

Revenue by rep and region:
Region     North  South   West
Sales_Rep                     
Alice      433.0    0.0    0.0
Bob          0.0  675.5    0.0
Cara         0.0    0.0  393.0


This can help compare results, but I would not decide someone needs training from revenue alone. Different reps might have different customers or regions.

## Exercise 2.1 - Spending over time

**My practice AI prompt:** Explain how to change a date column to datetime and group spending by day of the week.

In [5]:
# Work only with money spent (negative amounts)
spending = transactions_df[transactions_df["Amount"] < 0].copy()
spending["Date"] = pd.to_datetime(spending["Date"])
spending["Spent"] = -spending["Amount"]
spending["Day"] = spending["Date"].dt.day_name()
spending["Month"] = spending["Date"].dt.to_period("M").astype(str)

print("Spending by day:")
print(spending.groupby("Day")["Spent"].sum().sort_values(ascending=False))
print("\nMonthly spending:")
print(spending.groupby("Month")["Spent"].sum())
print("\nLargest single expense:")
print(spending.loc[spending["Spent"].idxmax(), ["Date", "Category", "Spent"]])

Spending by day:
Day
Sunday       285
Wednesday    213
Thursday     195
Friday        85
Saturday      60
Monday        45
Tuesday       40
Name: Spent, dtype: int64

Monthly spending:
Month
2026-07    235
2026-08    295
2026-09    393
Name: Spent, dtype: int64

Largest single expense:
Date        2026-09-27 00:00:00
Category               Shopping
Spent                       150
Name: 11, dtype: object


**My observation:** The monthly totals help me check whether spending is going up. There are only three months of made-up data here, so it would be better to use more data before calling anything a long-term trend.

## Exercise 2.2 - Budget by category

**My practice AI prompt:** How can I calculate average monthly spending by category and suggest a budget based on the data? Keep the calculations simple.

In [6]:
monthly = spending.groupby(["Month", "Category"])["Spent"].sum().unstack(fill_value=0)
average_monthly = monthly.mean().round(2)
variation = monthly.std().round(2)
# Add a small 10% buffer to the average monthly amount
suggested_budget = (average_monthly * 1.10).round(2)
budget_table = pd.DataFrame({
    "Average_Monthly": average_monthly,
    "Variation": variation,
    "Suggested_Budget": suggested_budget
}).sort_values("Average_Monthly", ascending=False)
print(budget_table)

           Average_Monthly  Variation  Suggested_Budget
Category                                               
Groceries           123.33      32.15            135.66
Shopping             80.00      75.50             88.00
Dining               60.00      25.00             66.00
Transport            44.33       4.04             48.76


The suggested budgets are just starting estimates using a 10% buffer. A category with high variation may need a bigger buffer. The standard deviation column shows how much monthly spending changes, but there is not enough data to make strong recommendations.

## Exercise 3 - Critiquing AI code

### Critique 1: Code that is too complex
The example uses several functions together, including a complicated `groupby`, `lambda`, and multi-level columns. I found that harder to follow than necessary. It also uses `df` in one place but `transactions_df` in another, which is confusing.

Can you rewrite this using clear variable names, a few separate steps, and basic pandas functions? Please explain what each step does.

### Critique 2: Not enough business context
The example uses `cat` and `amt`, but those are not the column names in our data. It also prints numbers without explaining what they mean.

Please use the actual `Category` and `Amount` columns, add a heading and sort the spending from highest to lowest. Also explain what the results mean for a budget.

In [7]:
# A simpler version of the category summary
category_spending = spending.groupby("Category")["Spent"].sum()
print("Spending by category (highest first):")
print(category_spending.sort_values(ascending=False).round(2))

Spending by category (highest first):
Category
Groceries    370
Shopping     240
Dining       180
Transport    133
Name: Spent, dtype: int64


## Exercise 4.1 - Retail store decisions

I treated each region as a store for this practice. I want to see where sales and product demand are highest.

In [8]:
store_product_sales = sales_df.pivot_table(
    index="Region", columns="Product", values="Quantity",
    aggfunc="sum", fill_value=0
)
print("Units sold by store and product:")
print(store_product_sales)
print("\nTotal revenue by store:")
print(sales_df.groupby("Region")["Total_Sale"].sum().sort_values(ascending=False))

Units sold by store and product:
Product  Desk Chair  Notebook  Pens  Printer Paper
Region                                            
North             1         8    18             18
South             3        15    20             14
West              2        12    30              0

Total revenue by store:
Region
South    675.5
North    433.0
West     393.0
Name: Total_Sale, dtype: float64


I would check which products sell the most in each store when planning inventory. I would not discontinue products or schedule promotions based on this small sample alone, because it does not show stock levels or demand over time.

## Exercise 4.2 - Simple finance advice

Show me how to calculate what percentage of spending goes to each category and print simple suggestions.

In [9]:
category_totals = spending.groupby("Category")["Spent"].sum()
total_spent = category_totals.sum()
percentages = (category_totals / total_spent * 100).round(1)
print("=== SPENDING BREAKDOWN ===")
for category, percentage in percentages.sort_values(ascending=False).items():
    print(f"{category}: {percentage:.1f}%")
    if percentage >= 30:
        print("  This is a large part of spending. It may be worth reviewing.")
    else:
        print("  This category is below 30% of total spending.")

=== SPENDING BREAKDOWN ===
Groceries: 40.1%
  This is a large part of spending. It may be worth reviewing.
Shopping: 26.0%
  This category is below 30% of total spending.
Dining: 19.5%
  This category is below 30% of total spending.
Transport: 14.4%
  This category is below 30% of total spending.



## Self-assessment

I practised writing prompts with clear questions, using pandas to group data.

## Reflection

The explanations of `groupby()` and converting dates were useful because those parts were new to me.

Asking one specific question and requesting simple code made the answer easier to understand.

I used clearer variable names and avoided the overly complicated code in the critique exercise.

I understand the basics better now, but I still need practice with pandas.

I will ask AI to explain the code, test the results and check whether the data actually supports the conclusion.